In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS, THEME_TO_TOPICS_MAPPING, CONCEPT_TO_CHAPTER_MAPPING

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "num_rows",
    "num_parse_errors",
    "num_gen_errors",
    #*PRED_COLS
]

def error_stats_table(jobs, cols):

    print(f"Processing {len(jobs)} jobs.")

    total_rows = 0
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")
        num_rows = len(df)
        total_rows += num_rows

        # LLM failed to produce proper json
        mask_no_errors = (df[PRED_COLS] != "\"PARSE ERROR\"").all(axis=1)
        df = df[mask_no_errors]
        num_parse_errors = list(mask_no_errors).count(False)

        # LLM did not follow instructions
        mask_generate_fail = df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        df = df[mask_generate_fail]
        num_gen_errors = list(mask_generate_fail).count(False)

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append error counts
        table["num_rows"].append(num_rows)
        table["num_parse_errors"].append(num_parse_errors)
        table["num_gen_errors"].append(num_gen_errors)

    return table, total_rows

def error_df(jobs, cols):

    print(f"Processing {len(jobs)} jobs.")

    job_ids = []
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")
        num_rows = len(df)

        # LLM failed to produce proper json
        mask_generate_fail = ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        df = df[mask_generate_fail]

        config = get_config(job["config_json"])

        # Append run configs
        config_cols = cols[:4]
        for col in config_cols:
            table[col] = table[col] + [config[col]] * len(df)

        # Append pred
        for col in PRED_COLS:
            table[col] = table[col] + list(df[col])

    return table

In [4]:
def get_suite(row):

    n_demos = row["number_of_demonstrations"]
    type_demos = row["type_of_demonstrations"][0:3]
    instr = "impl" if row["use_instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

In [5]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

errs, rows = error_stats_table(jobs_list, cols)
res = pd.DataFrame(errs)
res = prettify_table(res)

Processing 330 jobs.


In [6]:
errs = error_df(jobs_list, cols[:4] + PRED_COLS)

Processing 330 jobs.


In [7]:
pd.set_option('display.max_rows', 10)

err_df = pd.DataFrame(errs)

mask_parse_errors = (err_df[PRED_COLS] == "\"PARSE ERROR\"").any(axis=1)

#err_df[~mask_parse_errors]

In [8]:
res = res.sort_values(by=["use_instructions", "number_of_demonstrations", "type_of_demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

In [9]:
# Show all columns
pd.options.display.max_columns = None
# Set float formatting
pd.options.display.float_format = '{:,.3f}'.format
bycols = [
    "model",
    "suite"
]

agg = aggregate_results(res, bycols, cols[5:], funs=["sum"])
# Drop count
# Drop rows with no errors
error_rows = (agg.loc[:, :] > 0).any(axis=1)
agg = agg[error_rows]

In [10]:
#formatted = format_table(agg)
formatted = agg.copy()

#formatted.columns = formatted.columns.get_level_values(0) # Flatten

#formatted = formatted.loc[["num_parse_errors", "num_gen_errors"],:]

parse_errs_sum = formatted["num_parse_errors"].sum()
gen_errs_sum = formatted["num_gen_errors"].sum()

#formatted.loc[("Total sum", "")] = (parse_errs_sum, gen_errs_sum) # Add total
#formatted.loc[("Error rate", "", "", "")] = (parse_errs_sum / rows, gen_errs_sum / rows) # Add total

print(parse_errs_sum)
print(gen_errs_sum)

print(parse_errs_sum / rows)
print(gen_errs_sum / rows)

sum    1054
dtype: int64
sum    199
dtype: int64
sum   0.006
dtype: float64
sum   0.001
dtype: float64


In [11]:
formatted.groupby(by="model").agg("sum")

,num_parse_errors,num_gen_errors
,sum,sum
model,,
Llama-3.1-8B-Instruct,789,0
Llama-3.3-70B-Instruct,18,0
Mistral-7B-Instruct-v0.3,158,18
Mistral-Small-3.2-24B-Instruct-2506,88,0
Qwen3-32B,1,181


In [12]:
tables_location = "./latex/tables"
os.makedirs(tables_location, exist_ok=True)
with open(f"{tables_location}/err_table.tex", "w", encoding="UTF-8") as f:
    text = formatted.to_latex(escape=True, sparsify=True)
    print(text)
    #f.write(text)

\begin{tabular}{llrr}
\toprule
 &  & num\_parse\_errors & num\_gen\_errors \\
 &  & sum & sum \\
model & suite &  &  \\
\midrule
\multirow[t]{5}{*}{Llama-3.1-8B-Instruct} & 1-neg-impl & 5 & 0 \\
 & 1-pos-impl & 508 & 0 \\
 & 6-mix-impl & 54 & 0 \\
 & 6-neg-impl & 3 & 0 \\
 & 6-pos-impl & 219 & 0 \\
\cline{1-4}
\multirow[t]{3}{*}{Llama-3.3-70B-Instruct} & 1-neg-impl & 5 & 0 \\
 & 1-pos-impl & 11 & 0 \\
 & 6-pos-impl & 2 & 0 \\
\cline{1-4}
\multirow[t]{5}{*}{Mistral-7B-Instruct-v0.3} & 1-neg-impl & 121 & 0 \\
 & 1-pos-impl & 32 & 0 \\
 & 6-mix-impl & 1 & 0 \\
 & 6-neg-impl & 3 & 0 \\
 & 6-pos-impl & 1 & 18 \\
\cline{1-4}
\multirow[t]{9}{*}{Mistral-Small-3.2-24B-Instruct-2506} & 0-non-expl & 3 & 0 \\
 & 1-neg-impl & 22 & 0 \\
 & 1-pos-impl & 6 & 0 \\
 & 6-mix-expl & 5 & 0 \\
 & 6-mix-impl & 15 & 0 \\
 & 6-neg-expl & 7 & 0 \\
 & 6-neg-impl & 22 & 0 \\
 & 6-pos-expl & 1 & 0 \\
 & 6-pos-impl & 7 & 0 \\
\cline{1-4}
\multirow[t]{3}{*}{Qwen3-32B} & 1-neg-impl & 0 & 31 \\
 & 6-neg-expl & 1 & 0 \